Software Name: llm-acd-supplemental-material

SPDX-FileCopyrightText: Copyright (c) 2025 Orange SA

SPDX-License-Identifier: MIT


This software is distributed under the MIT license,
the text of which is available at https://opensource.org/license/MIT/
or see the "LICENSE" file for more details.


Authors: See CONTRIBUTORS.txt

Software description: The supplemental material for the paper From Zero-Shot to Reward-Aware: Evaluating Prompting and Memory in LLM-Based Cyber Defense Agents".

In [ ]:
import os
import re

import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

## Parsing functions

In [ ]:
def extract_episode_rewards_rl(log_file_path: str, adversary: str, steps: int, green: bool) -> list[float]:
    green_str = str(green)
    target_header = f"--- Variant: steps={steps}, adversary={adversary}, green={green_str} ---"

    rewards = []
    in_target_variant = False

    reward_pattern = re.compile(r"episode total reward:\s*([-\d.]+)")
    variant_pattern = re.compile(r"^--- Variant:")

    with open(log_file_path, "r") as f:
        for line in f:
            line = line.strip()

            if target_header in line:
                in_target_variant = True
                continue

            if in_target_variant and variant_pattern.match(line):
                break

            if in_target_variant:
                match = reward_pattern.search(line)
                if match:
                    if "iqm:" not in line:
                        rewards.append(float(match.group(1)))

    if not rewards:
        raise ValueError(
            f"No variant: "
            f"adversary={adversary}, steps={steps}, green={green}"
        )

    if len(rewards) != 100:
        print(f"Warning: {len(rewards)} episode.")

    return rewards

In [ ]:
def get_run_reward(log_file):

    res = []

    with open(log_file, 'r') as logs:
        for line in logs:
            if 'Reward at 30 steps:' in line:
                res.append(float(line[20:-1]))
            if 'Reward at 50 steps:' in line:
                res.append(float(line[20:-1]))
            if 'Reward at 100 steps:' in line:
                res.append(float(line[21:-1]))

    return res

In [ ]:
def get_list_reward(log_file):

  list_reward = []

  is_prompt_passed = False

  with open(log_file, 'r') as logs:
    for line in logs:
      if not is_prompt_passed and '='*200 in line:
        is_prompt_passed = True
      if is_prompt_passed and 'Reward:' in line and 'Final prompt' not in line:
        list_reward.append(float(line[8:-1]))

  return list_reward

In [ ]:
def get_nb_invalid_action(log_folder):
    total_action = 0
    count = 0
    for run_logs_file in os.listdir(log_folder):
        total_action += 100
        with open(log_folder + run_logs_file, 'r') as logs:
            lines = logs.readlines()
            for i, line in enumerate(lines):
                if "Blue Action (Invalid):" in line:
                    #print(run_logs_file)
                    count += 1

                    """
                    if i + 4 < len(lines):
                        response_line = lines[i + 4]
                        if "Response:" in response_line:
                            raw_output = response_line.split("Response:", 1)[1].strip()
                            try:
                                json_output = json.loads(raw_output)
                                print(json_output["content"][0]["text"])
                                print("---------------------------------------\n")
                            except (json.JSONDecodeError, KeyError, IndexError) as e:
                                print(f"Could not parse JSON output: {e}")
                    """

    print(f"Number of invalid action accross all runs: {count}; {count/total_action*100}%")

In [ ]:
def get_latence(log_folder):

  latences = []

  for run_logs_file in os.listdir(log_folder):
    with open(log_folder + run_logs_file, 'r') as logs:
        for line in logs:
            if 'Time:' in line:
                latences.append(float(line[6:-3]))

  return print(f"Mean action time: {np.array(latences).mean():.2f} ± {np.array(latences).std():.2f}, {min(latences):.2f} / {max(latences):.2f}, median {np.median(latences):.2f}")

## Stats functions

In [ ]:
def interquartile_mean(x, axis):
    q1 = np.percentile(x, 25, axis=axis, keepdims=True)
    q3 = np.percentile(x, 75, axis=axis, keepdims=True)
    mask = (x >= q1) & (x <= q3)
    return np.sum(x * mask, axis=axis) / np.sum(mask, axis=axis)

In [ ]:
def get_reward_stats_iqm(log_folder):
    # Get reward at timestep 30, 50 and 100 for each run
    tot_r30 = []
    tot_r50 = []
    tot_r100 = []
    for run_logs_file in os.listdir(log_folder):
        r30, r50, r100 = get_run_reward(log_folder + run_logs_file)
        tot_r30.append(r30)
        tot_r50.append(r50)
        tot_r100.append(r100)

    # Convert to np array
    tot_r30 = np.array(tot_r30)
    tot_r50 = np.array(tot_r50)
    tot_r100 = np.array(tot_r100)
    rewards = np.column_stack([tot_r30, tot_r50, tot_r100])

    # Get BCI for interquartile mean
    bcis = stats.bootstrap(
        (rewards,),
        interquartile_mean,
        axis=0,
        confidence_level=0.95,
        method="percentile",
        random_state=0,
    )

    # Calculate interquartile mean
    iqms = interquartile_mean(rewards, axis=0)

    # Get min-max
    mins = rewards.min(axis=0)
    maxs = rewards.max(axis=0)

    return iqms, bcis, mins, maxs

In [ ]:
def iqm_diff_statistic(a, b, axis):
    return interquartile_mean(a, axis=axis) - interquartile_mean(b, axis=axis)

In [ ]:
def permutation_testing(model_a, model_b):
    result = stats.permutation_test(
        data=(model_a, model_b),
        statistic=iqm_diff_statistic,
        permutation_type="independent",
        vectorized=True,
        n_resamples=10_000,
        alternative="two-sided",
        random_state=42,
    )
    return result.pvalue

In [ ]:
def holm_correction(pvalues):
    reject, p_adjusted, _, _ = multipletests(pvalues, alpha=0.05, method='holm')
    return reject, p_adjusted

## Utils

In [ ]:
def format_stat(iqms, bcis, mins, maxs):
    print(f"Reward:")
    print(f"30: {iqms[0]:.2f} [{bcis.confidence_interval.low[0]:.2f}, {bcis.confidence_interval.high[0]:.2f}], {mins[0]:.2f} -- {maxs[0]:.2f}")
    print(f"50: {iqms[1]:.2f} [{bcis.confidence_interval.low[1]:.2f}, {bcis.confidence_interval.high[1]:.2f}], {mins[1]:.2f} -- {maxs[1]:.2f}")
    print(f"100: {iqms[2]:.2f} [{bcis.confidence_interval.low[2]:.2f}, {bcis.confidence_interval.high[2]:.2f}], {mins[2]:.2f} -- {maxs[2]:.2f}")

## Section 5.1

Interquartile Mean, Boostrapped Condidence Interval and Range of LLM with prompt 2 against BLine over 30 runs.

In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("GPT-5")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Mini/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("GPT-5 Mini")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Nano/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("GPT-5-Nano")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/Claude-Sonnet-4.6/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Claude Sonnet 4.6")
format_stat(iqms, bcis, mins, maxs)
print("")


Latences of LLM.

In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5/'
print("GPT-5")
get_latence(log_folder)
print()

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Mini/'
print("GPT-5 Mini")
get_latence(log_folder)
print()

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Nano/'
print("GPT-5 Nano")
get_latence(log_folder)
print()

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/Claude-Sonnet-4.6/'
print("Claude Sonnet 4.6")
get_latence(log_folder)
print()

Number of invalid runs for LLM.

In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5/'
print("GPT-5")
get_nb_invalid_action(log_folder)
print()

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Mini/'
print("GPT-5 Mini")
get_nb_invalid_action(log_folder)
print()

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Nano/'
print("GPT-5 Nano")
get_nb_invalid_action(log_folder)
print()

log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/Claude-Sonnet-4.6/'
print("Claude Sonnet 4.6")
get_nb_invalid_action(log_folder)
print()

RL results are available in the the RL log file. They can also be reproduced by running the RL Agent notebook.

Permutation tests.

In [ ]:
llm_log_path = ''
rl_log_path = ''

# Extract GPT Results
gpt_nano = []
log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Nano/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    gpt_nano.append(r100)

gpt_mini = []
log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5-Mini/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    gpt_mini.append(r100)

gpt = []
log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/GPT-5/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    gpt.append(r100)

sonnet = []
log_folder = llm_log_path + '1 - Rl vs LLM Baseline Performance/Claude-Sonnet-4.6/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    sonnet.append(r100)

# Extract RL results
rl = extract_episode_rewards_rl(
    log_file_path=rl_log_path + "rl-complete.log",
    adversary="B_lineAgent",
    steps=100,
    green=False
)

# Permutation test
gpt_nano_rl = permutation_testing(gpt_nano, rl)
gpt_mini_rl = permutation_testing(gpt_mini, rl)
gpt_rl = permutation_testing(gpt, rl)
sonnet_rl = permutation_testing(sonnet, rl)

pvalues = [gpt_nano_rl, gpt_mini_rl, gpt_rl, sonnet_rl]
labels  = ["gpt_nano_rl", "gpt_mini_rl", "gpt_rl" , "sonnet_rl"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")
print("")

## Section 5.2

GPT-5 Results

In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/BLine - No Green/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("BLine - No Green")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/BLine - Green/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("BLine - Green")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/RedMeander - No Green/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("RedMeander - No Green")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/RedMeander - Green/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("RedMeander - Green")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/Sleep - No Green/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Sleep - No Green")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/Sleep - Green/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Sleep - Green")
format_stat(iqms, bcis, mins, maxs)
print("")

RL results are available in the the RL log file.

Permutation test

In [ ]:
rl_log_path = ''
llm_log_path = ''

# GPT generalization
print("GPT generalization")
bl_gpt = []
log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/BLine - No Green/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    bl_gpt.append(r100)

bl_g_gpt = []
log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/BLine - Green/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    bl_g_gpt.append(r100)

rm_gpt = []
log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/RedMeander - No Green/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    rm_gpt.append(r100)

rm_g_gpt = []
log_folder = llm_log_path + '2 - Generalization to Adversaries and Benign Traffic/RedMeander - Green/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    rm_g_gpt.append(r100)

bl_rm_gpt = permutation_testing(bl_gpt, rm_gpt)
blg_rmg_gpt = permutation_testing(bl_g_gpt, rm_g_gpt)
bl_blg_gpt = permutation_testing(bl_gpt, bl_g_gpt)
rm_rmg_gpt = permutation_testing(rm_gpt, rm_g_gpt)

pvalues = [bl_rm_gpt, blg_rmg_gpt, bl_blg_gpt, rm_rmg_gpt]
labels  = ["bl_rm_gpt", "blg_rmg_gpt", "bl_blg_gpt", "rm_rmg_gpt"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")
print("")

# RL generalization
print("RL generalization")

bl_rl = extract_episode_rewards_rl(
    log_file_path=rl_log_path + "rl-complete.log",
    adversary="B_lineAgent",
    steps=100,
    green=False
)

bl_g_rl = extract_episode_rewards_rl(
    log_file_path=rl_log_path + "rl-complete.log",
    adversary="B_lineAgent",
    steps=100,
    green=True
)

rm_rl = extract_episode_rewards_rl(
    log_file_path=rl_log_path + "rl-complete.log",
    adversary="RedMeanderAgent",
    steps=100,
    green=False
)

rm_g_rl = extract_episode_rewards_rl(
    log_file_path=rl_log_path + "rl-complete.log",
    adversary="RedMeanderAgent",
    steps=100,
    green=True
)

bl_rm_rl = permutation_testing(bl_rl, rm_rl)
blg_rmg_rl = permutation_testing(bl_g_rl, rm_g_rl)
bl_blg_rl = permutation_testing(bl_rl, bl_g_rl)
rm_rmg_rl = permutation_testing(rm_rl, rm_g_rl)

pvalues = [bl_rm_rl, blg_rmg_rl, bl_blg_rl, rm_rmg_rl]
labels  = ["bl_rm_rl", "blg_rmg_rl", "bl_blg_rl", "rm_rmg_rl"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")
print("")

# GPT vs RL
print("GPT vs RL")
bl_gpt_rl = permutation_testing(bl_gpt, bl_rl)
blg_gpt_rl = permutation_testing(bl_g_gpt, bl_g_rl)
rm_gpt_rl = permutation_testing(rm_gpt, rm_rl)
rmg_gpt_rl = permutation_testing(rm_g_gpt, rm_g_rl)

pvalues = [bl_gpt_rl, blg_gpt_rl, rm_gpt_rl, rmg_gpt_rl]
labels  = ["bl_gpt_rl", "blg_gpt_rl", "rm_gpt_rl" , "rmg_gpt_rl"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")
print("")

## Section 5.3

In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Base - BLine/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Base BLine")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Simpl - BLine/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Reduced BLine")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Aug - BLine/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Augmented BLine")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Base - RedMeander/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Base RedMeander")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Simpl - RedMeander/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Reduced RedMeander")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Aug - RedMeander/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Augmented RedMeander")
format_stat(iqms, bcis, mins, maxs)
print("")

Permutation tests.

In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Base - BLine/'
bl_base = []
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    bl_base.append(r100)

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Simpl - BLine/'
bl_simpl = []
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    bl_simpl.append(r100)

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Aug - BLine/'
bl_aug = []
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    bl_aug.append(r100)

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Base - RedMeander/'
rm_base = []
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    rm_base.append(r100)

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Simpl - RedMeander/'
rm_simpl = []
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    rm_simpl.append(r100)

log_folder = llm_log_path + '3 - Generalization to Network Topologies/Aug - RedMeander/'
rm_aug = []
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    rm_aug.append(r100)

bl_base_simpl = permutation_testing(bl_base, bl_simpl)
bl_base_aug = permutation_testing(bl_base, bl_aug)
rm_base_simpl = permutation_testing(rm_base, rm_simpl)
rm_base_aug = permutation_testing(rm_base, rm_aug)

pvalues = [bl_base_simpl, bl_base_aug, rm_base_simpl, rm_base_aug]
labels  = ["bl_base_simpl", "bl_base_aug", "rm_base_simpl", "rm_base_aug"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")

## Section 5.4

GPT-5 results for each prompt variant.


In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 1")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 2/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 2")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 3")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 4")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 5")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 6/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 6")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 7")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 8/GPT/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 8")
format_stat(iqms, bcis, mins, maxs)

Claude Sonnet 4.6 results for each prompt variant.

In [ ]:
llm_log_path = ''

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 1")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 2/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 2")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 3")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 4")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 5")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 6/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 6")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 7")
format_stat(iqms, bcis, mins, maxs)
print("")

log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 8/Sonnet/'
iqms, bcis, mins, maxs = get_reward_stats_iqm(log_folder)
print("Prompt 8")
format_stat(iqms, bcis, mins, maxs)

Permutation tests between zero-shot and few-shot prompts for GPT.

In [ ]:
llm_log_path = ''

p1_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p1_gpt.append(r100)

p2_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 2/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p2_gpt.append(r100)

p3_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_gpt.append(r100)

p4_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p4_gpt.append(r100)

p5_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p5_gpt.append(r100)

p6_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 6/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p6_gpt.append(r100)

p7_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p7_gpt.append(r100)

p8_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 8/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p8_gpt.append(r100)

p1_p2 = permutation_testing(p1_gpt, p2_gpt)
p3_p4 = permutation_testing(p3_gpt, p4_gpt)
p5_p6 = permutation_testing(p5_gpt, p6_gpt)
p7_p8 = permutation_testing(p7_gpt, p8_gpt)

pvalues = [p1_p2, p3_p4, p5_p6, p7_p8]
labels  = ["p1_p2", "p3_p4", "p5_p6", "p7_p8"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")

Permutation tests between zero-shot and few-shot prompts for Claude.

In [ ]:
llm_log_path = ''

p1_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p1_sonnet.append(r100)

p2_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 2/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p2_sonnet.append(r100)

p3_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_sonnet.append(r100)

p4_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p4_sonnet.append(r100)

p5_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p5_sonnet.append(r100)

p6_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 6/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p6_sonnet.append(r100)

p7_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p7_sonnet.append(r100)

p8_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 8/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p8_sonnet.append(r100)

p1_p2 = permutation_testing(p1_sonnet, p2_sonnet)
p3_p4 = permutation_testing(p3_sonnet, p4_sonnet)
p5_p6 = permutation_testing(p5_sonnet, p6_sonnet)
p7_p8 = permutation_testing(p7_sonnet, p8_sonnet)

pvalues = [p1_p2, p3_p4, p5_p6, p7_p8]
labels  = ["p1_p2", "p3_p4", "p5_p6", "p7_p8"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")

Permutation tests between claude and gpt in zero-shot.

In [ ]:
llm_log_path = ''

p1_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p1_claude.append(r100)

p1_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p1_gpt.append(r100)

p3_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_claude.append(r100)

p3_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_gpt.append(r100)

p5_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p5_claude.append(r100)

p5_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p5_gpt.append(r100)

p7_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p7_claude.append(r100)

p7_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p7_gpt.append(r100)

# Without feed back
p1 = permutation_testing(p1_claude, p1_gpt)
p3 = permutation_testing(p3_claude, p3_gpt)

pvalues = [p1, p3]
labels  = ["p1", "p3"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")

# With feedback
p5 = permutation_testing(p5_claude, p5_gpt)
p7 = permutation_testing(p7_claude, p7_gpt)

pvalues = [p5, p7]
labels  = ["p5", "p7"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")

Permutation tests between with and without memory.

In [ ]:
llm_log_path = ''

p1_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p1_claude.append(r100)

p1_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 1/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p1_gpt.append(r100)

p2_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 2/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p2_claude.append(r100)

p2_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 2/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p2_gpt.append(r100)

p3_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_claude.append(r100)

p3_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_gpt.append(r100)

p4_claude = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p4_claude.append(r100)

p4_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p4_gpt.append(r100)

# GPT
p1_p3_gpt = permutation_testing(p1_gpt, p3_gpt)
p2_p4_gpt = permutation_testing(p2_gpt, p4_gpt)

pvalues = [p1_p3_gpt, p2_p4_gpt]
labels  = ["p1_p3_gpt", "p2_p4_gpt"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")

# Claude
p1_p3_sonnet = permutation_testing(p1_claude, p3_claude)
p2_p4_sonnet = permutation_testing(p2_claude, p4_claude)

pvalues = [p1_p3_sonnet, p2_p4_sonnet]
labels  = ["p1_p3_sonnet", "p2_p4_sonnet"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")


Permutation test with and without feedback.

In [ ]:
llm_log_path = ''

# GPT
print("GPT")
p3_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_gpt.append(r100)

p5_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p5_gpt.append(r100)

p7_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p7_gpt.append(r100)

p4_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p4_gpt.append(r100)

p6_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 6/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p6_gpt.append(r100)

p8_gpt = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 8/GPT/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p8_gpt.append(r100)

p3_p5_gpt = permutation_testing(p3_gpt, p5_gpt)
p5_p7_gpt = permutation_testing(p5_gpt, p7_gpt)
p3_p7_gpt = permutation_testing(p3_gpt, p7_gpt)
p4_p6_gpt = permutation_testing(p4_gpt, p6_gpt)
p6_p8_gpt = permutation_testing(p6_gpt, p8_gpt)
p4_p8_gpt = permutation_testing(p4_gpt, p8_gpt)

pvalues = [p3_p5_gpt, p5_p7_gpt, p3_p7_gpt, p4_p6_gpt, p6_p8_gpt, p4_p8_gpt]
labels  = ["p3_p5_gpt", "p5_p7_gpt", "p3_p7_gpt", "p4_p6_gpt", "p6_p8_gpt", "p4_p8_gpt"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")
print("")


# Claude
print("Claude")
p3_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 3/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p3_sonnet.append(r100)

p5_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 5/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p5_sonnet.append(r100)

p7_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 7/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p7_sonnet.append(r100)

p4_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 4/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p4_sonnet.append(r100)

p6_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 6/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p6_sonnet.append(r100)

p8_sonnet = []
log_folder = llm_log_path + '4 - Impact of Prompt Strategy/Prompt 8/Sonnet/'
for run_logs_file in os.listdir(log_folder):
    _, _, r100 = get_run_reward(log_folder + run_logs_file)
    p8_sonnet.append(r100)

p3_p5_sonnet = permutation_testing(p3_sonnet, p5_sonnet)
p5_p7_sonnet = permutation_testing(p5_sonnet, p7_sonnet)
p3_p7_sonnet = permutation_testing(p3_sonnet, p7_sonnet)
p4_p6_sonnet = permutation_testing(p4_sonnet, p6_sonnet)
p6_p8_sonnet = permutation_testing(p6_sonnet, p8_sonnet)
p4_p8_sonnet = permutation_testing(p4_sonnet, p8_sonnet)

pvalues = [p3_p5_sonnet, p5_p7_sonnet, p3_p7_sonnet, p4_p6_sonnet, p6_p8_sonnet, p4_p8_sonnet]
labels  = ["p3_p5_sonnet", "p5_p7_sonnet", "p3_p7_sonnet", "p4_p6_sonnet", "p6_p8_sonnet", "p4_p8_sonnet"]

reject, p_adj = holm_correction(pvalues)

for name, p_raw, p_adj, sig in zip(labels, pvalues, p_adj, reject):
  print(f"{name:<5}  p_raw={p_raw:.4f}  p_adj={p_adj:.4f}  significant={sig}")